# Chapter 3 — Exercises: Worked Solutions

Worked solutions for Chapter 3 (Cubic Equations of State).

**Exercises:**
1. van der Waals critical point derivation
2. SRK vs PR density comparison
3. Volume translation effect on liquid density

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim\src\main\resources
  3. C:\Users\ESOL\.m2\repository\com\h2database\h2\2.4.240\h2-2.4.240.jar
  4. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-api\2.25.4\log4j-api-2.25.4.jar
  5. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-core\2.25.4\log4j-core-2.25.4.jar
  6. C:\Users\ESOL\.m2\repository\com\thoughtworks\xstream\xstream\1.4.21\xstream-1.4.21.jar
  7. C:\Users\ESOL\.m2\repository\io\github\x-stream\mxparser\1.2.2\mxparser-1.2.2.jar
  8. C:\Users\ESOL\.m2\repository\xmlpull\xmlpull\1.1.3.1\xmlpull-1.1.3.1.jar
  9. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-lang3\3.20.0\commons-lang3-3.20.0.jar
  10. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-math3\3.6.1\commons-math3-3.6.1.jar
  11. C:\Users\ESOL\.m2\repository\org\ejml\ejml-all\0.44.0\ejml-all-0.44.0.jar
  12


JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim mode: devtools


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
R = 8.314

---
## Exercise 3.1 — van der Waals Critical Point

**Problem:** Starting from the van der Waals equation:

$$P = \frac{RT}{v - b} - \frac{a}{v^2}$$

derive the critical point conditions and show that $Z_c = 3/8 = 0.375$.

### Solution

At the critical point, the first and second derivatives of pressure
with respect to volume vanish simultaneously:

$$\left(\frac{\partial P}{\partial v}\right)_T = 0, \quad \left(\frac{\partial^2 P}{\partial v^2}\right)_T = 0$$

Taking derivatives:

$$\frac{\partial P}{\partial v} = -\frac{RT}{(v-b)^2} + \frac{2a}{v^3} = 0$$

$$\frac{\partial^2 P}{\partial v^2} = \frac{2RT}{(v-b)^3} - \frac{6a}{v^4} = 0$$

Dividing the first equation by the second:

$$\frac{v-b}{2} = \frac{v}{3} \implies v_c = 3b$$

Substituting back:

$$T_c = \frac{8a}{27Rb}, \quad P_c = \frac{a}{27b^2}$$

Therefore:

$$Z_c = \frac{P_c v_c}{RT_c} = \frac{a \cdot 3b}{27b^2 \cdot R \cdot \frac{8a}{27Rb}} = \frac{3}{8} = 0.375$$

In [3]:
# Numerical verification: plot vdW isotherms near critical point
a_ex = 0.2283  # Pa m6/mol2 (methane-like)
b_ex = 4.278e-5  # m3/mol
Tc_vdw = 8 * a_ex / (27 * R * b_ex)
Pc_vdw = a_ex / (27 * b_ex**2)
vc_vdw = 3 * b_ex
Zc_vdw = Pc_vdw * vc_vdw / (R * Tc_vdw)

print(f"vdW critical point: Tc = {Tc_vdw:.1f} K, Pc = {Pc_vdw/1e5:.1f} bar, vc = {vc_vdw*1e6:.1f} cm3/mol")
print(f"Zc = {Zc_vdw:.4f} (should be 0.375)")

fig, ax = plt.subplots(figsize=(3.5, 3.0))
v_range = np.linspace(1.2 * b_ex, 8 * vc_vdw, 500)
for Tr, col, ls in [(0.9, BLUE, "--"), (1.0, ORANGE, "-"), (1.1, GREEN, "-.")]:
    T = Tr * Tc_vdw
    P = R * T / (v_range - b_ex) - a_ex / v_range**2
    ax.plot(v_range * 1e6, P / 1e5, color=col, linestyle=ls, linewidth=1.2,
            label=f"$T_r = {Tr}$")

ax.plot(vc_vdw * 1e6, Pc_vdw / 1e5, 'ko', markersize=6)
ax.annotate("Critical point", xy=(vc_vdw * 1e6, Pc_vdw / 1e5),
            xytext=(vc_vdw * 3e6, Pc_vdw * 1.2 / 1e5),
            fontsize=7, arrowprops=dict(arrowstyle="->", color="grey", lw=0.5))
ax.set_xlabel(r"Molar volume (cm$^3$/mol)")
ax.set_ylabel("Pressure (bar)")
ax.set_title(r"Exercise 3.1: vdW isotherms, $Z_c = 3/8$")
ax.set_ylim(0, Pc_vdw * 2 / 1e5)
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch03_ex01_vdw_critical.png")

vdW critical point: Tc = 190.2 K, Pc = 46.2 bar, vc = 128.3 cm3/mol
Zc = 0.3750 (should be 0.375)


Saved: fig_ch03_ex01_vdw_critical.png


**Answer:** $Z_c = 3/8 = 0.375$ for the vdW EoS. Real substances have
$Z_c \approx 0.23$–$0.29$, which is why more sophisticated cubic EoS
(SRK, PR) are needed. The SRK gives $Z_c = 1/3 \approx 0.333$ and
PR gives $Z_c \approx 0.307$, both closer to experimental values.

---
## Exercise 3.2 — SRK vs PR Liquid Density

**Problem:** Calculate the liquid density of $n$-octane at 1 bar from
20°C to the normal boiling point using SRK and PR. Compare with NIST data.
Which EoS is more accurate for liquid density?

### Solution

In [4]:
if NEQSIM_MODE == "pip":
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    SystemPrEos = jneqsim.thermo.system.SystemPrEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
else:
    from neqsim import jneqsim
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    SystemPrEos = jneqsim.thermo.system.SystemPrEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

temps = np.arange(20, 126, 5)  # C (BP of octane ~ 125 C)
rho_srk, rho_pr = [], []

for T_C in temps:
    for cls, mr, rho_list in [(SystemSrkEos, "classic", rho_srk),
                               (SystemPrEos, "classic", rho_pr)]:
        try:
            f = cls(273.15 + float(T_C), 1.013)
            f.addComponent("n-octane", 1.0)
            f.setMixingRule(mr)
            ops = ThermodynamicOperations(f)
            ops.TPflash()
            f.initProperties()
            rho = float(f.getPhase(1).getDensity("kg/m3"))
            rho_list.append(rho)
        except Exception:
            rho_list.append(np.nan)

# NIST reference data for n-octane liquid at 1 atm
nist_T = [20, 40, 60, 80, 100, 120]
nist_rho = [703, 684, 664, 643, 621, 596]  # kg/m3

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.0, 2.8))
ax1.plot(temps[:len(rho_srk)], rho_srk, color=BLUE, linewidth=1.5, label="SRK")
ax1.plot(temps[:len(rho_pr)], rho_pr, color=ORANGE, linewidth=1.5, linestyle="--", label="PR")
ax1.scatter(nist_T, nist_rho, color="black", s=25, zorder=5, label="NIST")
ax1.set_xlabel("Temperature (\u00b0C)"); ax1.set_ylabel("Density (kg/m\u00b3)")
ax1.set_title("(a) $n$-Octane liquid density")
ax1.legend(frameon=True, fontsize=7); ax1.grid(True, alpha=0.3)

# Errors
err_srk = [(np.interp(t, temps, rho_srk) - r) / r * 100 for t, r in zip(nist_T, nist_rho)]
err_pr = [(np.interp(t, temps, rho_pr) - r) / r * 100 for t, r in zip(nist_T, nist_rho)]
x = np.arange(len(nist_T))
ax2.bar(x - 0.15, err_srk, 0.3, color=BLUE, alpha=0.7, label="SRK")
ax2.bar(x + 0.15, err_pr, 0.3, color=ORANGE, alpha=0.7, label="PR")
ax2.axhline(y=0, color="black", linewidth=0.5)
ax2.set_xticks(x); ax2.set_xticklabels([f"{t}\u00b0C" for t in nist_T], fontsize=7)
ax2.set_xlabel("Temperature"); ax2.set_ylabel("Error (%)")
ax2.set_title("(b) Density error vs NIST")
ax2.legend(frameon=True, fontsize=7); ax2.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch03_ex02_srk_pr_density.png")

Saved: fig_ch03_ex02_srk_pr_density.png


**Answer:** PR typically gives better liquid density predictions than SRK
because it was designed with a critical compressibility of $Z_c = 0.307$,
closer to typical experimental values. SRK ($Z_c = 0.333$) systematically
under-predicts liquid densities by 5–15%. Volume translation (Péneloux
correction) can fix both, but PR starts from a better baseline.

---
## Exercise 3.3 — Péneloux Volume Translation

**Problem:** The Péneloux volume translation shifts the molar volume:

$$v_{\text{corrected}} = v_{\text{EoS}} - c$$

For propane with $c = 5.2$ cm$^3$/mol, compute the corrected liquid
density at 25\u00b0C, 10 bar and compare with the uncorrected SRK value
and experimental data ($\rho_{\text{exp}} = 493$ kg/m$^3$).

### Solution

In [5]:
T_C, P_bar = 25.0, 10.0
MW_propane = 44.096  # g/mol
c_peneloux = 5.2e-6  # m3/mol (5.2 cm3/mol)
rho_exp = 493.0  # kg/m3

f = SystemSrkEos(273.15 + T_C, P_bar)
f.addComponent("propane", 1.0)
f.setMixingRule("classic")
ops = ThermodynamicOperations(f)
ops.TPflash()
f.initProperties()

v_srk = float(f.getPhase(1).getMolarVolume()) / 1e5  # convert to m3/mol
rho_srk_val = MW_propane / 1000 / v_srk  # kg/m3

v_corrected = v_srk - c_peneloux
rho_corrected = MW_propane / 1000 / v_corrected

print(f"SRK molar volume:       {v_srk*1e6:.1f} cm3/mol")
print(f"Corrected molar volume: {v_corrected*1e6:.1f} cm3/mol")
print(f"\nSRK density:       {rho_srk_val:.1f} kg/m3  (error: {(rho_srk_val-rho_exp)/rho_exp*100:.1f}%)")
print(f"Corrected density: {rho_corrected:.1f} kg/m3  (error: {(rho_corrected-rho_exp)/rho_exp*100:.1f}%)")
print(f"Experimental:      {rho_exp:.1f} kg/m3")

SRK molar volume:       97.6 cm3/mol
Corrected molar volume: 92.4 cm3/mol

SRK density:       451.6 kg/m3  (error: -8.4%)
Corrected density: 477.0 kg/m3  (error: -3.2%)
Experimental:      493.0 kg/m3


**Answer:** The Péneloux correction significantly improves the liquid
density prediction. The correction is a constant shift in volume space,
meaning it does not affect vapor–liquid equilibrium (VLE) calculations
but substantially improves volumetric properties. This is the basis for
the volume-translated SRK (VT-SRK) commonly used in process simulation.